# Detecting Bangla DeepFake Audio Using LSTM

**Reproduction Study** — Based on the paper `Detecting Bangla Deepfake Using Lstm and WaveNet` by Ayan et al. (2026)

This notebook replicates the methodology from the paper using the paper's own Bangla audio dataset. We will call it `LSTM Bangla Audio Dataset`. The dataset contains both real and deepfake audio samples in Bangla.
We implement one model: **RNN-based LSTM** using MFCC features.
The goal of this notebook is to reproduce the results reported in the paper and to provide a clear implementation of the LSTM model for Bangla deepfake audio detection.

---

## 1. Imports and Configuration

In [2]:
import os
import glob
import random
import warnings
import json
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import librosa
import librosa.display
import IPython.display as ipd
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

# ------------------------------------------------------------------
# NOTE: scikit-learn is intentionally NOT used.
# This keeps the notebook easy to run on Python 3.13 without compiling
# scikit-learn from source. We implement the few needed utilities here.
# ------------------------------------------------------------------

def stratified_split_indices(y, test_size=0.2, seed=42):
    """Return (train_idx, test_idx) with stratification by class."""
    y = np.asarray(y)
    rng = np.random.default_rng(seed)

    all_idx = np.arange(len(y))
    train_parts = []
    test_parts = []

    classes = np.unique(y)
    for cls in classes:
        cls_idx = all_idx[y == cls]
        rng.shuffle(cls_idx)

        if len(cls_idx) <= 1:
            n_test = 0
        else:
            n_test = int(round(len(cls_idx) * float(test_size)))
            n_test = max(1, min(len(cls_idx) - 1, n_test))

        test_parts.append(cls_idx[:n_test])
        train_parts.append(cls_idx[n_test:])

    test_idx = np.concatenate(test_parts) if len(test_parts) else np.array([], dtype=int)
    train_idx = np.concatenate(train_parts) if len(train_parts) else np.array([], dtype=int)

    rng.shuffle(test_idx)
    rng.shuffle(train_idx)
    return train_idx, test_idx


def confusion_matrix_np(y_true, y_pred, num_classes=2):
    y_true = np.asarray(y_true, dtype=int)
    y_pred = np.asarray(y_pred, dtype=int)
    cm = np.zeros((num_classes, num_classes), dtype=np.int64)
    for t, p in zip(y_true, y_pred):
        cm[t, p] += 1
    return cm


def _per_class_stats_from_cm(cm):
    tp = np.diag(cm).astype(np.float64)
    fp = cm.sum(axis=0).astype(np.float64) - tp
    fn = cm.sum(axis=1).astype(np.float64) - tp
    support = cm.sum(axis=1).astype(np.float64)

    precision = np.divide(tp, tp + fp, out=np.zeros_like(tp), where=(tp + fp) != 0)
    recall = np.divide(tp, tp + fn, out=np.zeros_like(tp), where=(tp + fn) != 0)
    f1 = np.divide(2 * precision * recall, precision + recall, out=np.zeros_like(tp), where=(precision + recall) != 0)

    return precision, recall, f1, support


def accuracy_score_np(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    return float(np.mean(y_true == y_pred))


def weighted_precision_recall_f1_np(y_true, y_pred, num_classes=2):
    cm = confusion_matrix_np(y_true, y_pred, num_classes=num_classes)
    precision, recall, f1, support = _per_class_stats_from_cm(cm)
    total = support.sum() if support.sum() > 0 else 1.0
    w = support / total
    return float((precision * w).sum()), float((recall * w).sum()), float((f1 * w).sum())


def classification_report_np(y_true, y_pred, target_names=None, digits=4):
    y_true = np.asarray(y_true, dtype=int)
    y_pred = np.asarray(y_pred, dtype=int)

    num_classes = int(max(y_true.max(initial=0), y_pred.max(initial=0)) + 1)
    if target_names is None:
        target_names = [str(i) for i in range(num_classes)]

    cm = confusion_matrix_np(y_true, y_pred, num_classes=num_classes)
    precision, recall, f1, support = _per_class_stats_from_cm(cm)

    acc = accuracy_score_np(y_true, y_pred)
    w_prec, w_rec, w_f1 = weighted_precision_recall_f1_np(y_true, y_pred, num_classes=num_classes)

    fmt = f"{{:>{digits+6}.{digits}f}}"
    lines = []
    lines.append("".ljust(16) + "precision".rjust(12) + "recall".rjust(12) + "f1-score".rjust(12) + "support".rjust(12))

    for i, name in enumerate(target_names):
        lines.append(
            name.ljust(16)
            + fmt.format(precision[i]).rjust(12)
            + fmt.format(recall[i]).rjust(12)
            + fmt.format(f1[i]).rjust(12)
            + f"{int(support[i]):12d}"
        )

    lines.append("")
    lines.append("accuracy".ljust(16) + "".rjust(12) + "".rjust(12) + fmt.format(acc).rjust(12) + f"{int(support.sum()):12d}")
    lines.append("weighted avg".ljust(16) + fmt.format(w_prec).rjust(12) + fmt.format(w_rec).rjust(12) + fmt.format(w_f1).rjust(12) + f"{int(support.sum()):12d}")

    return "\n".join(lines)


def roc_curve_np(y_true, y_score):
    """Binary ROC curve (y_true in {0,1}). Returns (fpr, tpr, thresholds)."""
    y_true = np.asarray(y_true, dtype=int)
    y_score = np.asarray(y_score, dtype=np.float64)

    order = np.argsort(-y_score, kind="mergesort")
    y_true = y_true[order]
    y_score = y_score[order]

    distinct = np.where(np.diff(y_score))[0]
    threshold_idxs = np.r_[distinct, y_true.size - 1]

    tps = np.cumsum(y_true)[threshold_idxs]
    fps = 1 + threshold_idxs - tps

    tps = np.r_[0, tps]
    fps = np.r_[0, fps]

    P = y_true.sum()
    N = y_true.size - P

    tpr = tps / P if P > 0 else np.zeros_like(tps, dtype=np.float64)
    fpr = fps / N if N > 0 else np.zeros_like(fps, dtype=np.float64)

    thresholds = np.r_[np.inf, y_score[threshold_idxs]]
    return fpr, tpr, thresholds


def auc_np(x, y):
    x = np.asarray(x, dtype=np.float64)
    y = np.asarray(y, dtype=np.float64)
    return float(np.trapz(y, x))


# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {DEVICE}")

# Plot style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('Set2')
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 11

Using device: cuda


In [3]:
# ============================================================
# Configuration — Kaggle input/output + ablation setup
# ============================================================
IS_KAGGLE = os.path.exists('/kaggle/input')
OUTPUT_ROOT = '/kaggle/working/lstm_ablation_outputs' if IS_KAGGLE else './lstm_ablation_outputs'
os.makedirs(OUTPUT_ROOT, exist_ok=True)

REAL_FOLDERS = ['banglafake_real', 'lstm_real']
FAKE_FOLDERS = ['baglafake_deepfake', 'crikk_deepfake', 'gemini_deepfake', 'lstm_deepfake']
ALL_REQUIRED_FOLDERS = REAL_FOLDERS + FAKE_FOLDERS

def _norm_name(name):
    return ''.join(ch for ch in name.lower() if ch.isalnum())

def _resolve_folder_aliases(root_dir, expected_names):
    if not os.path.isdir(root_dir):
        return None

    children = [d for d in os.listdir(root_dir) if os.path.isdir(os.path.join(root_dir, d))]
    children_norm = {_norm_name(d): d for d in children}

    aliases = {
        'baglafake_deepfake': ['baglafake_deepfake', 'banglafake_deepfake'],
        'banglafake_real': ['banglafake_real', 'baglafake_real'],
        'crikk_deepfake': ['crikk_deepfake'],
        'gemini_deepfake': ['gemini_deepfake'],
        'lstm_deepfake': ['lstm_deepfake'],
        'lstm_real': ['lstm_real'],
    }

    resolved = {}
    for exp in expected_names:
        candidates = aliases.get(exp, [exp])
        found = None
        for c in candidates:
            c_norm = _norm_name(c)
            if c_norm in children_norm:
                found = os.path.join(root_dir, children_norm[c_norm])
                break
        if found is None:
            return None
        resolved[exp] = found

    return resolved

def resolve_data_root(required_folders):
    candidates = [
        './Data',
        './Bangla Audio DeepFake/Data',
        '/kaggle/working/Data',
        '/kaggle/input/Data',
    ]

    for c in candidates:
        mapped = _resolve_folder_aliases(c, required_folders)
        if mapped is not None:
            return os.path.abspath(c), mapped

    if os.path.isdir('/kaggle/input'):
        for ds in sorted(glob.glob('/kaggle/input/*')):
            mapped = _resolve_folder_aliases(ds, required_folders)
            if mapped is not None:
                return os.path.abspath(ds), mapped

            for root, dirs, _ in os.walk(ds):
                rel = os.path.relpath(root, ds)
                depth = 0 if rel == '.' else rel.count(os.sep) + 1
                if depth > 3:
                    dirs[:] = []
                    continue
                if os.path.basename(root).lower() == 'data':
                    mapped = _resolve_folder_aliases(root, required_folders)
                    if mapped is not None:
                        return os.path.abspath(root), mapped

    raise FileNotFoundError(
        'Could not locate Data folder with required subfolders. '
        'Please verify Kaggle extraction path and folder names.'
    )

DATA_ROOT, RESOLVED_DATA_FOLDERS = resolve_data_root(ALL_REQUIRED_FOLDERS)
print(f'Data root: {DATA_ROOT}')
print(f'Output root: {os.path.abspath(OUTPUT_ROOT)}')

# 5 experiments total: 4 leave-one-fake-out ablations + 1 full stratified split
ALL_EXPERIMENTS = [
    {'name': 'holdout_crikk', 'holdout_fake': 'crikk_deepfake', 'split_type': 'holdout'},
    {'name': 'holdout_lstm', 'holdout_fake': 'lstm_deepfake', 'split_type': 'holdout'},
    {'name': 'holdout_gemini', 'holdout_fake': 'gemini_deepfake', 'split_type': 'holdout'},
    {'name': 'holdout_baglafake', 'holdout_fake': 'baglafake_deepfake', 'split_type': 'holdout'},
    {'name': 'full_dataset_stratified', 'split_type': 'stratified_full'},
]

# Toggle 1: 'all' or 'one'
RUN_EXPERIMENT_MODE = 'all'

# Toggle 2: used only when RUN_EXPERIMENT_MODE == 'one'
SINGLE_EXPERIMENT_NAME = 'holdout_crikk'

valid_names = [e['name'] for e in ALL_EXPERIMENTS]
if RUN_EXPERIMENT_MODE not in ['all', 'one']:
    raise ValueError("RUN_EXPERIMENT_MODE must be 'all' or 'one'")
if RUN_EXPERIMENT_MODE == 'all':
    EXPERIMENTS = ALL_EXPERIMENTS
else:
    if SINGLE_EXPERIMENT_NAME not in valid_names:
        raise ValueError(f'Invalid SINGLE_EXPERIMENT_NAME: {SINGLE_EXPERIMENT_NAME}. Choose from: {valid_names}')
    EXPERIMENTS = [e for e in ALL_EXPERIMENTS if e['name'] == SINGLE_EXPERIMENT_NAME]

# Audio processing
SAMPLE_RATE      = 16000
AUDIO_DURATION   = 5
N_MFCC           = 40
MAX_LEN_MFCC     = 157

# Used for full_dataset_stratified experiment
TEST_SIZE        = 0.20

# Training
VAL_SIZE         = 0.10
BATCH_SIZE       = 64
LSTM_EPOCHS      = 40
LSTM_LR          = 1e-4
NUM_CLASSES      = 2

print(f'Run mode: {RUN_EXPERIMENT_MODE}')
print(f'Experiments to run: {[e["name"] for e in EXPERIMENTS]}')

Data root: f:\ML Project\Bangla Audio Deepfake Detection\Ablation Study\Data
Output root: f:\ML Project\Bangla Audio Deepfake Detection\Ablation Study\lstm_ablation_outputs
Run mode: all
Experiments to run: ['holdout_crikk', 'holdout_lstm', 'holdout_gemini', 'holdout_baglafake', 'full_dataset_stratified']


## 2. Data Loading and Exploration

In [4]:
def collect_audio_files(folder_path):
    """Collect audio files recursively with case-insensitive extension handling."""
    if not os.path.isdir(folder_path):
        return []

    exts = {'.wav', '.flac', '.mp3', '.m4a', '.ogg'}
    files = []
    for root, _, fnames in os.walk(folder_path):
        for fn in fnames:
            if os.path.splitext(fn)[1].lower() in exts:
                files.append(os.path.join(root, fn))

    return sorted(files)


def build_dataset_df(data_root, exclude_fake=None):
    records = []
    use_fake_folders = [f for f in FAKE_FOLDERS if f != exclude_fake]

    for folder in REAL_FOLDERS:
        folder_path = RESOLVED_DATA_FOLDERS[folder]
        audio_files = collect_audio_files(folder_path)
        for fp in audio_files:
            records.append({
                'filepath': fp,
                'label': 0,
                'label_name': 'Real',
                'source': folder,
            })

    for folder in use_fake_folders:
        folder_path = RESOLVED_DATA_FOLDERS[folder]
        audio_files = collect_audio_files(folder_path)
        for fp in audio_files:
            records.append({
                'filepath': fp,
                'label': 1,
                'label_name': 'Fake',
                'source': folder,
            })

    if len(records) == 0:
        raise RuntimeError(f'No audio files found under resolved paths from {data_root}')

    return pd.DataFrame(records)

# Preview with full dataset
df = build_dataset_df(DATA_ROOT, exclude_fake=None)
print(f'Total samples: {len(df)}')
print('\nLabel distribution:')
print(df['label_name'].value_counts())
print('\nSources used:')
print(df['source'].value_counts())

Total samples: 7311

Label distribution:
label_name
Fake    3911
Real    3400
Name: count, dtype: int64

Sources used:
source
banglafake_real       2400
lstm_real             1000
baglafake_deepfake    1000
crikk_deepfake        1000
lstm_deepfake         1000
gemini_deepfake        911
Name: count, dtype: int64


In [ ]:
# --- Class distribution bar chart ---
fig, ax = plt.subplots(1, 1, figsize=(6, 4.5))

# Overall distribution
counts = df['label_name'].value_counts()
ax.bar(counts.index, counts.values, color=['#2ecc71', '#e74c3c'], edgecolor='black')
ax.set_title('Overall Class Distribution')
ax.set_ylabel('Count')
for i, v in enumerate(counts.values):
    ax.text(i, v + 100, str(v), ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig('class_distribution.png', bbox_inches='tight')
plt.show()

## 3. Audio Visualization

In [5]:
def load_audio(filepath, sr=SAMPLE_RATE, duration=AUDIO_DURATION):
    """Load audio file, resample to target sr, pad/truncate to fixed duration."""
    y, sr_orig = librosa.load(filepath, sr=sr, duration=duration)
    target_len = sr * duration
    if len(y) < target_len:
        y = np.pad(y, (0, target_len - len(y)), mode='constant')
    else:
        y = y[:target_len]
    return y, sr


def get_sample_speaker(row):
    """Return speaker if present; otherwise fall back to source folder."""
    speaker = row.get('speaker', None)
    if speaker is None or (isinstance(speaker, float) and np.isnan(speaker)) or str(speaker).strip() == '':
        return row.get('source', 'unknown')
    return speaker


# Pick one real and one fake sample for visualization
sample_real = df[df['label'] == 0].sample(1, random_state=SEED).iloc[0]
sample_fake = df[df['label'] == 1].sample(1, random_state=SEED).iloc[0]

y_real, _ = load_audio(sample_real['filepath'])
y_fake, _ = load_audio(sample_fake['filepath'])

real_speaker = get_sample_speaker(sample_real)
fake_speaker = get_sample_speaker(sample_fake)

print(f"Real sample: {os.path.basename(sample_real['filepath'])} (speaker/source: {real_speaker})")
print(f"Fake sample: {os.path.basename(sample_fake['filepath'])} (speaker/source: {fake_speaker})")

Real sample: deepfake_data_mozilla__real_wav__common_voice_s3_292.wav (speaker/source: banglafake_real)
Fake sample: crikk_0552_Nabanita_354.mp3 (speaker/source: crikk_deepfake)


In [ ]:
# --- Waveform, Histogram, Spectrogram, and MFCC for real vs. fake ---
fig, axes = plt.subplots(4, 2, figsize=(15, 16))

for col, (y, title) in enumerate([(y_real, 'Real Audio'), (y_fake, 'Fake Audio')]):
    # Waveform
    librosa.display.waveshow(y, sr=SAMPLE_RATE, ax=axes[0, col], color='#3498db' if col == 0 else '#e74c3c')
    axes[0, col].set_title(f'Waveform — {title}')
    axes[0, col].set_xlabel('Time (s)')
    axes[0, col].set_ylabel('Amplitude')

    # Histogram
    axes[1, col].hist(y, bins=100, color='#3498db' if col == 0 else '#e74c3c', alpha=0.8, edgecolor='black')
    axes[1, col].set_title(f'Amplitude Histogram — {title}')
    axes[1, col].set_xlabel('Amplitude')
    axes[1, col].set_ylabel('Frequency')

    # Mel Spectrogram
    S = librosa.feature.melspectrogram(y=y, sr=SAMPLE_RATE, n_mels=128)
    S_dB = librosa.power_to_db(S, ref=np.max)
    img = librosa.display.specshow(S_dB, sr=SAMPLE_RATE, x_axis='time', y_axis='mel', ax=axes[2, col])
    axes[2, col].set_title(f'Mel Spectrogram — {title}')
    fig.colorbar(img, ax=axes[2, col], format='%+2.0f dB')

    # MFCC
    mfcc = librosa.feature.mfcc(y=y, sr=SAMPLE_RATE, n_mfcc=N_MFCC)
    img2 = librosa.display.specshow(mfcc, sr=SAMPLE_RATE, x_axis='time', ax=axes[3, col])
    axes[3, col].set_title(f'MFCC — {title}')
    fig.colorbar(img2, ax=axes[3, col])

plt.tight_layout()
plt.savefig('audio_visualization.png', bbox_inches='tight')
plt.show()

In [ ]:
# --- Listen to samples ---
print("Real Audio:")
ipd.display(ipd.Audio(y_real, rate=SAMPLE_RATE))
print("\nFake Audio:")
ipd.display(ipd.Audio(y_fake, rate=SAMPLE_RATE))

## 4. Feature Extraction

### 4a. MFCC Features (for LSTM model)

As described in the paper, MFCC features are extracted from each audio file.
We extract **40 MFCC coefficients** per time frame and pad/truncate to a fixed number of time steps.

In [ ]:
def extract_mfcc(filepath, sr=SAMPLE_RATE, duration=AUDIO_DURATION,
                 n_mfcc=N_MFCC, max_len=MAX_LEN_MFCC):
    """Extract MFCC features from an audio file."""
    y, _ = load_audio(filepath, sr=sr, duration=duration)
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=n_mfcc)  # (n_mfcc, time)
    # Pad or truncate time axis
    if mfcc.shape[1] < max_len:
        mfcc = np.pad(mfcc, ((0, 0), (0, max_len - mfcc.shape[1])), mode='constant')
    else:
        mfcc = mfcc[:, :max_len]
    return mfcc.T  # (time, n_mfcc) — sequence-first for LSTM


# Quick test
test_mfcc = extract_mfcc(sample_real['filepath'])
print(f"MFCC shape : {test_mfcc.shape}  — (time_steps, n_mfcc)")

In [ ]:
# ============================================================
# Extract MFCC features for the full dataset
# ============================================================
print("Extracting MFCC features...")

mfcc_features = []
labels = []
valid_indices = []
valid_filepaths = []

for idx, row in tqdm(df.iterrows(), total=len(df), desc="Processing audio"):
    try:
        mfcc = extract_mfcc(row['filepath'])
        mfcc_features.append(mfcc)
        labels.append(row['label'])
        valid_indices.append(idx)
        valid_filepaths.append(row['filepath'])
    except Exception as e:
        print(f"  Skipping {row['filepath']}: {e}")

X_mfcc = np.array(mfcc_features, dtype=np.float32)
del mfcc_features  # free the list immediately
y_all = np.array(labels, dtype=np.int64)
filepaths_all = np.array(valid_filepaths)

print(f"\nExtracted features for {len(y_all)} samples")
print(f"MFCC array shape : {X_mfcc.shape}")
print(f"Label distribution: Real={np.sum(y_all == 0)}, Fake={np.sum(y_all == 1)}")

## 5. Stratified Train / Validation / Test Split

Following the paper's methodology, we use stratified splitting to maintain class balance across sets.

In [ ]:
# --- First split: train+val vs. test (80/20), stratified ---
train_val_idx, test_idx = stratified_split_indices(y_all, test_size=TEST_SIZE, seed=SEED)

# --- Second split: train vs. val from the train+val portion, stratified ---
relative_train_idx, relative_val_idx = stratified_split_indices(y_all[train_val_idx], test_size=VAL_SIZE, seed=SEED)
train_idx = train_val_idx[relative_train_idx]
val_idx = train_val_idx[relative_val_idx]

# MFCC splits (small enough to stay in RAM: ~600 MB)
X_train_mfcc, X_val_mfcc, X_test_mfcc = X_mfcc[train_idx], X_mfcc[val_idx], X_mfcc[test_idx]
y_train, y_val, y_test = y_all[train_idx], y_all[val_idx], y_all[test_idx]

# Free the unsplit array now that we have the splits
del X_mfcc
import gc; gc.collect()

print(f"Train : {len(y_train)} samples  (Real={np.sum(y_train==0)}, Fake={np.sum(y_train==1)})")
print(f"Val   : {len(y_val)} samples  (Real={np.sum(y_val==0)}, Fake={np.sum(y_val==1)})")
print(f"Test  : {len(y_test)} samples  (Real={np.sum(y_test==0)}, Fake={np.sum(y_test==1)})")

In [ ]:
# --- Split distribution visualization ---
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, (split_y, name) in zip(axes, [(y_train, 'Train'), (y_val, 'Validation'), (y_test, 'Test')]):
    unique, counts = np.unique(split_y, return_counts=True)
    label_names = ['Real', 'Fake']
    colors = ['#2ecc71', '#e74c3c']
    ax.bar(label_names, counts, color=colors, edgecolor='black')
    ax.set_title(f'{name} Set ({len(split_y)} samples)')
    ax.set_ylabel('Count')
    for i, v in enumerate(counts):
        ax.text(i, v + 20, str(v), ha='center', fontweight='bold')
plt.tight_layout()
plt.savefig('split_distribution.png', bbox_inches='tight')
plt.show()

## 6. PyTorch Datasets and DataLoaders

In [ ]:
class MFCCDataset(Dataset):
    """Dataset for MFCC features (LSTM model)."""
    def __init__(self, X, y):
        # Use from_numpy to share memory with the numpy array (no copy)
        self.X = torch.from_numpy(X)
        self.y = torch.from_numpy(y).long()

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


# MFCC DataLoaders
train_mfcc_ds = MFCCDataset(X_train_mfcc, y_train)
val_mfcc_ds   = MFCCDataset(X_val_mfcc, y_val)
test_mfcc_ds  = MFCCDataset(X_test_mfcc, y_test)

train_mfcc_dl = DataLoader(train_mfcc_ds, batch_size=BATCH_SIZE, shuffle=True,  drop_last=False)
val_mfcc_dl   = DataLoader(val_mfcc_ds,   batch_size=BATCH_SIZE, shuffle=False, drop_last=False)
test_mfcc_dl  = DataLoader(test_mfcc_ds,  batch_size=BATCH_SIZE, shuffle=False, drop_last=False)

print(f"MFCC — Train batches: {len(train_mfcc_dl)}, Val: {len(val_mfcc_dl)}, Test: {len(test_mfcc_dl)}")

---
## 7. RNN-Based LSTM Model

Following the paper:
- Input: MFCC features (time_steps, 40)
- LSTM layers with Tanh activation (built-in)
- Dropout rate = 0.5
- Dense layers with ReLU
- Output: Softmax over 2 classes
- Optimizer: Adam, lr = 0.0001

In [ ]:
class LSTMModel(nn.Module):
    """
    RNN-based LSTM model for deepfake audio detection using MFCC features.
    Architecture follows the paper's description:
      - Bidirectional LSTM layers
      - Dropout (0.5)
      - Fully connected layers with ReLU
      - Softmax output
    """
    def __init__(self, input_size=N_MFCC, hidden_size=128, num_layers=2,
                 num_classes=NUM_CLASSES, dropout=0.5):
        super(LSTMModel, self).__init__()

        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0,
            bidirectional=True,
        )
        self.dropout = nn.Dropout(dropout)
        self.fc1 = nn.Linear(hidden_size * 2, 64)   # *2 for bidirectional
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(64, num_classes)

    def forward(self, x):
        # x: (batch, time_steps, n_mfcc)
        lstm_out, (h_n, c_n) = self.lstm(x)
        # Use the last time step output
        out = lstm_out[:, -1, :]  # (batch, hidden*2)
        out = self.dropout(out)
        out = self.relu(self.fc1(out))
        out = self.dropout(out)
        out = self.fc2(out)
        return out  # raw logits — use CrossEntropyLoss


lstm_model = LSTMModel().to(DEVICE)
print(lstm_model)
total_params = sum(p.numel() for p in lstm_model.parameters() if p.requires_grad)
print(f"\nTotal trainable parameters: {total_params:,}")

### 7a. LSTM Training Loop

In [ ]:
def train_one_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    for X_batch, y_batch in dataloader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * X_batch.size(0)
        _, predicted = torch.max(outputs, 1)
        correct += (predicted == y_batch).sum().item()
        total += y_batch.size(0)

    epoch_loss = running_loss / total
    epoch_acc = correct / total
    return epoch_loss, epoch_acc


@torch.no_grad()
def evaluate(model, dataloader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    for X_batch, y_batch in dataloader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)

        running_loss += loss.item() * X_batch.size(0)
        _, predicted = torch.max(outputs, 1)
        correct += (predicted == y_batch).sum().item()
        total += y_batch.size(0)

    epoch_loss = running_loss / total
    epoch_acc = correct / total
    return epoch_loss, epoch_acc

In [ ]:
def extract_dataset_mfcc(df_in):
    mfcc_features, labels, filepaths, sources = [], [], [], []
    for _, row in tqdm(df_in.iterrows(), total=len(df_in), desc='Extracting MFCC'):
        try:
            mfcc = extract_mfcc(row['filepath'])
            mfcc_features.append(mfcc)
            labels.append(row['label'])
            filepaths.append(row['filepath'])
            sources.append(row['source'])
        except Exception as e:
            print(f"Skipping {row['filepath']}: {e}")

    X = np.array(mfcc_features, dtype=np.float32)
    y = np.array(labels, dtype=np.int64)
    fps = np.array(filepaths)
    src = np.array(sources)
    return X, y, fps, src

@torch.no_grad()
def get_predictions(model, dataloader, device):
    model.eval()
    all_preds, all_labels, all_probs = [], [], []
    for X_batch, y_batch in dataloader:
        X_batch = X_batch.to(device)
        outputs = model(X_batch)
        probs = torch.softmax(outputs, dim=1)
        _, predicted = torch.max(outputs, 1)
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(y_batch.numpy())
        all_probs.extend(probs.cpu().numpy())
    return np.array(all_labels), np.array(all_preds), np.array(all_probs)

def plot_training_curves(history, out_path):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    ax1.plot(history['train_loss'], label='Train Loss', linewidth=2)
    ax1.plot(history['val_loss'], label='Val Loss', linewidth=2)
    ax1.set_title('LSTM Loss Curves')
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Loss')
    ax1.legend()
    ax1.grid(True, alpha=0.3)

    ax2.plot(history['train_acc'], label='Train Accuracy', linewidth=2)
    ax2.plot(history['val_acc'], label='Val Accuracy', linewidth=2)
    ax2.set_title('LSTM Accuracy Curves')
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('Accuracy')
    ax2.legend()
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(out_path, bbox_inches='tight')
    plt.close(fig)

def split_leave_one_fake_out(X_all, y_all, fps_all, src_all, holdout_fake, val_size=0.10, seed=42):
    """Train on 3 fake sources; evaluate on held-out fake source + matched-ratio real samples."""
    idx_all = np.arange(len(y_all))
    idx_real = idx_all[y_all == 0]
    idx_fake = idx_all[y_all == 1]

    idx_fake_test = idx_fake[src_all[idx_fake] == holdout_fake]
    idx_fake_train = idx_fake[src_all[idx_fake] != holdout_fake]

    if len(idx_fake_test) == 0 or len(idx_fake_train) == 0:
        raise ValueError(f'Invalid holdout setup for {holdout_fake}: empty fake train/test split.')

    fake_train_ratio = len(idx_fake_train) / float(len(idx_fake_train) + len(idx_fake_test))
    n_real_total = len(idx_real)
    n_real_train = int(round(n_real_total * fake_train_ratio))
    n_real_train = max(1, min(n_real_total - 1, n_real_train))

    rng = np.random.default_rng(seed)
    idx_real_shuf = idx_real.copy()
    rng.shuffle(idx_real_shuf)
    idx_real_train = idx_real_shuf[:n_real_train]
    idx_real_test = idx_real_shuf[n_real_train:]

    idx_train_pool = np.concatenate([idx_fake_train, idx_real_train])
    idx_holdout_test = np.concatenate([idx_fake_test, idx_real_test])

    rel_train, rel_val = stratified_split_indices(y_all[idx_train_pool], test_size=val_size, seed=seed)
    idx_train = idx_train_pool[rel_train]
    idx_val = idx_train_pool[rel_val]

    return idx_train, idx_val, idx_holdout_test

def split_full_dataset_stratified(y_all, val_size=0.10, test_size=0.20, seed=42):
    """Standard full-dataset stratified split: train/val/test."""
    idx_all = np.arange(len(y_all))
    idx_train_val_rel, idx_test_rel = stratified_split_indices(y_all, test_size=test_size, seed=seed)
    idx_train_val = idx_all[idx_train_val_rel]
    idx_test = idx_all[idx_test_rel]

    rel_train, rel_val = stratified_split_indices(y_all[idx_train_val], test_size=val_size, seed=seed)
    idx_train = idx_train_val[rel_train]
    idx_val = idx_train_val[rel_val]
    return idx_train, idx_val, idx_test

def run_lstm_experiment(exp_cfg):
    exp_name = exp_cfg['name']
    split_type = exp_cfg.get('split_type', 'holdout')
    holdout_fake = exp_cfg.get('holdout_fake')

    if split_type == 'holdout':
        print(f"\n{'='*80}\nExperiment: {exp_name} | holdout_fake={holdout_fake}\n{'='*80}")
    else:
        print(f"\n{'='*80}\nExperiment: {exp_name} | split_type=stratified_full\n{'='*80}")

    exp_dir = os.path.join(OUTPUT_ROOT, exp_name)
    model_dir = os.path.join(exp_dir, 'models')
    plot_dir = os.path.join(exp_dir, 'plots')
    pred_dir = os.path.join(exp_dir, 'predictions')
    result_dir = os.path.join(exp_dir, 'results')
    for d in [exp_dir, model_dir, plot_dir, pred_dir, result_dir]:
        os.makedirs(d, exist_ok=True)

    df_exp = build_dataset_df(DATA_ROOT, exclude_fake=None)
    X_all, y_all, filepaths_all, source_all = extract_dataset_mfcc(df_exp)

    if split_type == 'holdout':
        idx_train, idx_val, idx_test = split_leave_one_fake_out(
            X_all, y_all, filepaths_all, source_all,
            holdout_fake=holdout_fake, val_size=VAL_SIZE, seed=SEED
        )
        test_split_name = 'holdout_test'
    elif split_type == 'stratified_full':
        idx_train, idx_val, idx_test = split_full_dataset_stratified(
            y_all, val_size=VAL_SIZE, test_size=TEST_SIZE, seed=SEED
        )
        test_split_name = 'stratified_test'
    else:
        raise ValueError(f'Unknown split_type: {split_type}')

    X_train, X_val, X_test = X_all[idx_train], X_all[idx_val], X_all[idx_test]
    y_train, y_val, y_test = y_all[idx_train], y_all[idx_val], y_all[idx_test]
    fp_test = filepaths_all[idx_test]
    src_test = source_all[idx_test]

    train_ds = MFCCDataset(X_train, y_train)
    val_ds = MFCCDataset(X_val, y_val)
    test_ds = MFCCDataset(X_test, y_test)

    train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=False)
    val_dl = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, drop_last=False)
    test_dl = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, drop_last=False)

    model = LSTMModel().to(DEVICE)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=LSTM_LR)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=5, factor=0.5)

    history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
    best_val_acc = 0.0
    best_model_path = os.path.join(model_dir, 'best_lstm_model.pth')

    for epoch in range(1, LSTM_EPOCHS + 1):
        train_loss, train_acc = train_one_epoch(model, train_dl, criterion, optimizer, DEVICE)
        val_loss, val_acc = evaluate(model, val_dl, criterion, DEVICE)
        scheduler.step(val_loss)

        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), best_model_path)

        if epoch % 5 == 0 or epoch == 1:
            print(f"Epoch {epoch:3d}/{LSTM_EPOCHS} | Train Loss {train_loss:.4f} Acc {train_acc:.4f} | Val Loss {val_loss:.4f} Acc {val_acc:.4f}")

    plot_training_curves(history, os.path.join(plot_dir, 'lstm_training_curves.png'))

    model.load_state_dict(torch.load(best_model_path, map_location=DEVICE, weights_only=True))
    y_true, y_pred, y_prob = get_predictions(model, test_dl, DEVICE)

    test_acc = accuracy_score_np(y_true, y_pred)
    test_prec, test_rec, test_f1 = weighted_precision_recall_f1_np(y_true, y_pred, num_classes=2)
    cm = confusion_matrix_np(y_true, y_pred, num_classes=2)

    fig, ax = plt.subplots(figsize=(7, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['Real', 'Fake'], yticklabels=['Real', 'Fake'],
                annot_kws={'size': 14}, ax=ax)
    if split_type == 'holdout':
        ax.set_title(f'LSTM Holdout Test Confusion Matrix ({exp_name})')
    else:
        ax.set_title(f'LSTM Stratified Test Confusion Matrix ({exp_name})')
    ax.set_xlabel('Predicted')
    ax.set_ylabel('True')
    plt.tight_layout()
    plt.savefig(os.path.join(plot_dir, 'lstm_confusion_matrix.png'), bbox_inches='tight')
    plt.close(fig)

    fpr, tpr, _ = roc_curve_np(y_true, y_prob[:, 1])
    test_auc = auc_np(fpr, tpr)
    fig, ax = plt.subplots(figsize=(8, 7))
    ax.plot(fpr, tpr, color='#3498db', linewidth=2, label=f'LSTM (AUC = {test_auc:.4f})')
    ax.plot([0, 1], [0, 1], 'k--', alpha=0.5)
    ax.set_xlabel('False Positive Rate')
    ax.set_ylabel('True Positive Rate')
    if split_type == 'holdout':
        ax.set_title(f'LSTM Holdout Test ROC Curve ({exp_name})')
    else:
        ax.set_title(f'LSTM Stratified Test ROC Curve ({exp_name})')
    ax.legend(loc='lower right')
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(plot_dir, 'lstm_roc_curve.png'), bbox_inches='tight')
    plt.close(fig)

    pred_df = pd.DataFrame({
        'filepath': fp_test,
        'source': src_test,
        'true_label': y_true,
        'pred_label': y_pred,
        'prob_real': y_prob[:, 0],
        'prob_fake': y_prob[:, 1],
    })
    if split_type == 'holdout':
        pred_df.to_csv(os.path.join(pred_dir, 'holdout_test_predictions.csv'), index=False)
    else:
        pred_df.to_csv(os.path.join(pred_dir, 'stratified_test_predictions.csv'), index=False)
    pred_df.to_csv(os.path.join(pred_dir, 'test_predictions.csv'), index=False)

    n_train_fake = int(np.sum(y_train == 1))
    n_test_fake = int(np.sum(y_test == 1))
    n_train_real = int(np.sum(y_train == 0))
    n_test_real = int(np.sum(y_test == 0))

    summary = {
        'experiment': exp_name,
        'split_type': split_type,
        'test_split_name': test_split_name,
        'holdout_fake_source': holdout_fake if split_type == 'holdout' else '',
        'dataset_size': int(len(df_exp)),
        'train_size': int(len(y_train)),
        'val_size': int(len(y_val)),
        'test_size': int(len(y_test)),
        'train_fake': n_train_fake,
        'test_fake': n_test_fake,
        'train_real': n_train_real,
        'test_real': n_test_real,
        'real_train_test_ratio': float(n_train_real / max(n_test_real, 1)),
        'fake_train_test_ratio': float(n_train_fake / max(n_test_fake, 1)),
        'best_val_accuracy': float(best_val_acc),
        'test_accuracy': float(test_acc),
        'test_precision': float(test_prec),
        'test_recall': float(test_rec),
        'test_f1': float(test_f1),
        'test_auc': float(test_auc),
    }

    if split_type == 'holdout':
        summary['holdout_test_size'] = int(len(y_test))
        summary['holdout_test_accuracy'] = float(test_acc)
        summary['holdout_test_precision'] = float(test_prec)
        summary['holdout_test_recall'] = float(test_rec)
        summary['holdout_test_f1'] = float(test_f1)
        summary['holdout_test_auc'] = float(test_auc)
    else:
        summary['stratified_test_size'] = int(len(y_test))
        summary['stratified_test_accuracy'] = float(test_acc)
        summary['stratified_test_precision'] = float(test_prec)
        summary['stratified_test_recall'] = float(test_rec)
        summary['stratified_test_f1'] = float(test_f1)
        summary['stratified_test_auc'] = float(test_auc)

    with open(os.path.join(result_dir, 'summary.json'), 'w') as f:
        json.dump(summary, f, indent=2)
    pd.DataFrame([summary]).to_csv(os.path.join(result_dir, 'summary.csv'), index=False)

    split_df = pd.DataFrame([
        {'split': 'train', 'real': int(np.sum(y_train == 0)), 'fake': int(np.sum(y_train == 1))},
        {'split': 'val', 'real': int(np.sum(y_val == 0)), 'fake': int(np.sum(y_val == 1))},
        {'split': test_split_name, 'real': int(np.sum(y_test == 0)), 'fake': int(np.sum(y_test == 1))},
    ])
    split_df.to_csv(os.path.join(result_dir, 'split_counts.csv'), index=False)

    if split_type == 'holdout':
        test_metrics_df = pd.DataFrame([{
            'experiment': exp_name,
            'holdout_fake_source': holdout_fake,
            'holdout_test_size': len(y_test),
            'holdout_test_accuracy': float(test_acc),
            'holdout_test_precision': float(test_prec),
            'holdout_test_recall': float(test_rec),
            'holdout_test_f1': float(test_f1),
            'holdout_test_auc': float(test_auc),
        }])
        test_metrics_df.to_csv(os.path.join(result_dir, 'holdout_test_metrics.csv'), index=False)
    else:
        test_metrics_df = pd.DataFrame([{
            'experiment': exp_name,
            'stratified_test_size': len(y_test),
            'stratified_test_accuracy': float(test_acc),
            'stratified_test_precision': float(test_prec),
            'stratified_test_recall': float(test_rec),
            'stratified_test_f1': float(test_f1),
            'stratified_test_auc': float(test_auc),
        }])
        test_metrics_df.to_csv(os.path.join(result_dir, 'stratified_test_metrics.csv'), index=False)

    print(classification_report_np(y_true, y_pred, target_names=['Real', 'Fake'], digits=4))
    if split_type == 'holdout':
        print(f"Holdout test metrics -- Acc: {test_acc:.4f} | AUC: {test_auc:.4f} | F1: {test_f1:.4f}")
    else:
        print(f"Stratified test metrics -- Acc: {test_acc:.4f} | AUC: {test_auc:.4f} | F1: {test_f1:.4f}")
    print(f"Split counts:\n{split_df.to_string(index=False)}")
    print(f"Saved artifacts to: {exp_dir}")
    return summary

MULTI_EXPERIMENT_MODE = True
all_summaries = []
for exp in EXPERIMENTS:
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)
    all_summaries.append(run_lstm_experiment(exp))

all_results_df = pd.DataFrame(all_summaries)
all_results_path = os.path.join(OUTPUT_ROOT, 'all_experiments_summary.csv')
all_results_df.to_csv(all_results_path, index=False)
print('\nAll LSTM experiments completed (4 holdout + 1 full stratified).')
print(f'Combined summary saved: {all_results_path}')

summary_cols = [
    'experiment', 'split_type', 'test_split_name', 'holdout_fake_source', 'test_size', 'test_accuracy', 'test_auc',
    'holdout_test_accuracy', 'holdout_test_auc', 'stratified_test_accuracy', 'stratified_test_auc',
    'train_real', 'test_real', 'train_fake', 'test_fake'
 ]
available_cols = [c for c in summary_cols if c in all_results_df.columns]
display(all_results_df[available_cols])

In [ ]:
if globals().get('MULTI_EXPERIMENT_MODE', False):
    print('Skipped: training curves are generated inside the multi-experiment runner cell.')

### 7b. LSTM Evaluation on Test Set

In [ ]:
if globals().get('MULTI_EXPERIMENT_MODE', False):
    print('Skipped: prediction helper is integrated in the multi-experiment runner cell.')

In [ ]:
if globals().get('MULTI_EXPERIMENT_MODE', False):
    print('Skipped: test evaluation is handled inside the multi-experiment runner cell.')

In [ ]:
if globals().get('MULTI_EXPERIMENT_MODE', False):
    print('Skipped: confusion matrix is generated inside each experiment output folder.')

---
## 9. Comparison of Results

We now compare our LSTM model against the paper's reported LSTM results:
- **Paper LSTM**: 89% accuracy

In [ ]:
if globals().get('MULTI_EXPERIMENT_MODE', False):
    print('Skipped: comparison table is included in combined experiment summary CSV.')

In [ ]:
if globals().get('MULTI_EXPERIMENT_MODE', False):
    print('Skipped: confusion matrix comparison plot is covered by per-experiment outputs.')

In [ ]:
if globals().get('MULTI_EXPERIMENT_MODE', False):
    print('Skipped: grouped comparison chart is covered by combined ablation summary.')

In [ ]:
if globals().get('MULTI_EXPERIMENT_MODE', False):
    print('Skipped: ROC plotting is generated for each experiment in the ablation runner.')

---
## 10. Inference on Individual Samples

In [ ]:
if globals().get('MULTI_EXPERIMENT_MODE', False):
    print('Skipped: single-sample inference section is disabled in multi-experiment mode.')

---
## 11. Summary and Conclusion

In [ ]:
if globals().get('MULTI_EXPERIMENT_MODE', False):
    print('Skipped: final summary text is replaced by per-experiment and combined summary files.')

In [ ]:
if globals().get('MULTI_EXPERIMENT_MODE', False):
    print('Skipped: final comparison figure is replaced by ablation summaries and per-experiment plots.')